# Кольцевые коммуникации

Реализуем три операции на CPU с Gloo. `rank` — номер процесса, `world_size` — число процессов. Входные тензоры одномерные и одинакового размера на всех процессах; их не меняем. Для reduce-scatter длина тензора должна делиться на число процессов, чтобы блоки были равными.

Каждый процесс отправляет блок соседу справа и получает блок слева. В reduce-scatter складываем полученные значения; индексы выбраны так, чтобы через `world_size - 1` раундов процесс получил сумму блока с номером `rank`. В all-gather за столько же раундов собираем все блоки в исходном порядке. All-reduce последовательно вызывает эти две операции и возвращает сумму.

Отправку делаем неблокирующей (`isend`), затем принимаем данные и ждём завершения отправки. Так процессы не застрянут в ожидании друг друга на отправке.


In [2]:
%%writefile collectives.py
import torch
import torch.distributed as dist


def ring_reduce_scatter(tensor):
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    left = (rank - 1) % world_size
    right = (rank + 1) % world_size
    chunks = [chunk.clone() for chunk in tensor.chunk(world_size)]
    received = torch.empty_like(chunks[rank])

    for step in range(world_size - 1):
        send_index = (rank - step - 1) % world_size
        recv_index = (rank - step - 2) % world_size
        request = dist.isend(chunks[send_index], dst=right)
        dist.recv(received, src=left)
        request.wait()
        chunks[recv_index].add_(received)

    return chunks[rank]


def ring_all_gather(shard):
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    left = (rank - 1) % world_size
    right = (rank + 1) % world_size
    chunks = [torch.empty_like(shard) for _ in range(world_size)]
    chunks[rank].copy_(shard)

    for step in range(world_size - 1):
        send_index = (rank - step) % world_size
        recv_index = (rank - step - 1) % world_size
        request = dist.isend(chunks[send_index], dst=right)
        dist.recv(chunks[recv_index], src=left)
        request.wait()

    return torch.cat(chunks)


def ring_all_reduce(tensor):
    return ring_all_gather(ring_reduce_scatter(tensor))


Overwriting collectives.py


## Сравнение с PyTorch

Сравним нашу реализацию с готовыми функциями торча `dist.reduce_scatter_tensor`, `dist.all_gather`
и `dist.all_reduce` на нескольких размерах тензоров. 

In [3]:
%%writefile check.py
from datetime import timedelta

import torch
import torch.distributed as dist

from collectives import ring_all_gather, ring_all_reduce, ring_reduce_scatter


torch.set_num_threads(1)
dist.init_process_group("gloo", timeout=timedelta(seconds=30))
try:
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    generator = torch.Generator().manual_seed(42 + rank)

    for chunk_size in (1, 7, 31):
        values = torch.randn(world_size * chunk_size, generator=generator, dtype=torch.float64)
        original = values.clone()
        expected_shard = torch.empty(chunk_size, dtype=values.dtype)
        dist.reduce_scatter_tensor(expected_shard, values.clone(), op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_reduce_scatter(values), expected_shard)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

        shard = torch.randn(chunk_size, generator=generator, dtype=torch.float64)
        original_shard = shard.clone()
        expected_parts = [torch.empty_like(shard) for _ in range(world_size)]
        dist.all_gather(expected_parts, shard)
        torch.testing.assert_close(ring_all_gather(shard), torch.cat(expected_parts))
        torch.testing.assert_close(shard, original_shard, rtol=0, atol=0)

        expected_sum = values.clone()
        dist.all_reduce(expected_sum, op=dist.ReduceOp.SUM)
        torch.testing.assert_close(ring_all_reduce(values), expected_sum)
        torch.testing.assert_close(values, original, rtol=0, atol=0)

    if rank == 0:
        print(f"OK: все три операции прошли проверки на {world_size} процессах.")
finally:
    dist.destroy_process_group()

Writing check.py


In [4]:
import sys

!{sys.executable} -m torch.distributed.run --standalone --nproc-per-node=2 check.py


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[Gloo] Rank 1 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
[Gloo] Rank 0 is connected to 1 peer ranks. Expected number of connected peer ranks is : 1
OK: все три операции прошли проверки на 2 процессах.


In [5]:
!{sys.executable} -m torch.distributed.run --standalone --nproc-per-node=4 check.py


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[Gloo] Rank 1 is connected to 3 peer ranks. Expected number of connected peer ranks is : 3
[Gloo] Rank 2 is connected to 3 peer ranks. Expected number of connected peer ranks is : 3
[Gloo] Rank 0 is connected to 3 peer ranks. Expected number of connected peer ranks is : 3
[Gloo] Rank 3 is connected to 3 peer ranks. Expected number of connected peer ranks is : 3
OK: все три операции прошли проверки на 4 процессах.
